<div dir="rtl">

# الدرس 47: الانتباه متعدد الرؤوس، وترتيب الكلمات

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### الانتباه متعدد الرؤوس

</div>

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.d_k = n_heads, d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)      # W_Q, W_K, W_V in one matrix
        self.out = nn.Linear(d_model, d_model)          # W_O

    def forward(self, x, causal=True):
        B, n, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)          # each (B, n, d)
        # split into heads: (B, n, d) -> (B, h, n, d_k)
        q, k, v = (t.view(B, n, self.h, self.d_k).transpose(1, 2) for t in (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=causal)   # (B, h, n, d_k)
        y = y.transpose(1, 2).reshape(B, n, d)          # merge heads back
        return self.out(y)

torch.manual_seed(0)
mha = MultiHeadAttention(d_model=768, n_heads=12)
x = torch.randn(2, 10, 768)
print("output shape:", tuple(mha(x).shape))
print("parameters  :", sum(p.numel() for p in mha.parameters()))
ref = nn.MultiheadAttention(embed_dim=768, num_heads=12)
print("PyTorch's own layer:", sum(p.numel() for p in ref.parameters()))

<div dir="rtl">

### الانتباه لا يعرف الترتيب: تجربة

</div>

In [ ]:
small = MultiHeadAttention(d_model=16, n_heads=4)
x = torch.randn(1, 5, 16)
perm = torch.tensor([3, 0, 4, 1, 2])
with torch.no_grad():
    y = small(x, causal=False)
    y_perm = small(x[:, perm], causal=False)
print("outputs of the shuffled input == shuffled outputs:", torch.allclose(y_perm, y[:, perm], atol=1e-6))

<div dir="rtl">

### الترميز الجيبي

</div>

In [ ]:
def sinusoidal(n_positions, d):
    pos = np.arange(n_positions)[:, None]                 # (n, 1)
    omega = 1 / 10000 ** (np.arange(0, d, 2) / d)         # (d/2,)
    pe = np.zeros((n_positions, d))
    pe[:, 0::2] = np.sin(pos * omega)                      # even dimensions
    pe[:, 1::2] = np.cos(pos * omega)                      # odd dimensions
    return pe

print(sinusoidal(3, 4).round(3))

In [ ]:
import matplotlib.pyplot as plt
pe = sinusoidal(100, 64)
plt.figure(figsize=(8, 3.4))
plt.imshow(pe.T, cmap="RdBu_r", aspect="auto", vmin=-1, vmax=1)
plt.xlabel("position"); plt.ylabel("dimension"); plt.colorbar(fraction=0.03)
plt.show()

<div dir="rtl">

### الترميز الدوراني: المسافة فقط

</div>

In [ ]:
def rotate(v, position, theta=0.1):
    a = position * theta
    R = np.array([[np.cos(a), -np.sin(a)], [np.sin(a), np.cos(a)]])
    return R @ v

q, k = np.array([1.0, 2.0]), np.array([0.5, -1.0])
near_start = rotate(q, 3) @ rotate(k, 1)          # positions 3 and 1: distance 2
far_away = rotate(q, 503) @ rotate(k, 501)        # positions 503 and 501: distance 2
other = rotate(q, 3) @ rotate(k, 0)               # distance 3
print(round(near_start, 6), round(far_away, 6), round(other, 6))

<div dir="rtl">

## تمرين: ترميز موضعين، ثم تشابههما

1. احسب بيدك الترميز الجيبي بأربعة أبعاد للموضع 3.
2. احسب الضرب النقطي بين ترميزي الموضعين 1 و 2، ثم بين 1 و 3. أيهما أكبر؟ ماذا يعني ذلك؟
3. تحقق بالكود.

</div>

In [ ]:
# اكتب حلّك هنا
